# Setup

In [ ]:
library(data.table)
options(datatable.na.strings=c('NA',''))

dir.create('data/raw/phenotypes',rec=T)
system('gcloud storage cp gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/raw/genomics/dosages.csv data/raw/genomics')

# Join all datasets

In [ ]:
data <-
Reduce(\(x,l) merge(x, l$y, by=l$by, all.x=T),
  list(               x=fread('data/derived/phenotypes/fhs_picsure_variables.csv'),
    list(by='NWD_Id', y=fread('data/raw/genomics/dosages.csv')                                      ),
    list(by='NWD_Id', y=fread('data/raw/phenotypes/freeze9_pcair_results.tsv')[, NWD_Id := sample.id]),
    list(by='TOM_Id', y=fread('data/derived/metabolomics/QCd/merged_QCd-aligned.csv')[grep('FHS',TOM_Id)][,TOM_Id:=sub('.*TOM','TOM',TOM_Id)])) # Merge metabolomics last b/c it adds tons of cols
)

# Rename columns

In [ ]:
data <- data |> setnames(\(nm)  sub('^PC','gPC', nm))# 'PC#' -> 'gPC#' To distinguish genetic PCs from metabolite PCs we'll add later
data <- data |> setnames('SEX','sex')

## Winsorization, imputation, transformation

In [ ]:
winsor      <- \(v, bounds=quantile(v,c(0,0.9),na.rm=T)) pmin(pmax(v,bounds[1]),bounds[2])
na_outliers <- \(v, bounds=quantile(v,c(0,0.9),na.rm=T)) fifelse(v<bounds[1] | v>bounds[2], NA, v)
nafill2 <- \(x) c(NA,x[!is.na(x)])[cumsum(!is.na(x))+1] # nafill(type='locf') but can handle non-numeric

data <- data[
  # (Median) imputation of covariates if not present in any exam (i.e. not present in exam 1)
  # (Making sure to take the median of the original set of non-missing values, NOT recalculating the median after each imputation!)
  ][, {age_median <<- median(age, na.rm=T);
       bmi_median <<- median(bmi, na.rm=T);
       .SD}

  ][ is.na(age), age := age_median
  ][ is.na(bmi), bmi := bmi_median

  # Definitions
  ][, hdl_log   := log(hdl)
  ][,  tg_log   := log(tg )
]

# Remap kinship matrix ids

In [ ]:
dir.create('data/raw/genomics',rec=T)
dir.create('data/derived/genomics',rec=T)

if(!file.exists('data/raw/genomics/km.Rdata')) system('gcloud storage cp gs://fc-secure-c5905035-bf75-471b-8fa8-fb8a7f83c4a9/submissions/e81b3733-44cb-4c17-87db-2aee99a0491a/fetch_dbgap/d64cc7fd-47b7-4352-985f-3c3951b9c3cf/call-decrypt/glob-b3ddbe8d141590fbae0db21546878fa2/km.Rdata data/raw/genomics')
load('data/raw/genomics/km.Rdata')
km <- km[rownames(km) %in% data$NWD_Id,
         colnames(km) %in% data$NWD_Id]

save(km, file='data/derived/genomics/fhs_km.RData')

## Write

In [ ]:
fwrite(data, 'data/derived/analysis_df-fhs.csv')

In [ ]:
system('gcloud storage cp data/derived/analysis_df-fhs.csv   gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/derived/analysis_df-fhs.csv')
system('gcloud storage cp data/derived/genomics/fhs_km.RData gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/derived/genomics/fhs_km.RData')